# PhiState A100 base-model training comparison

Select **Runtime → Change runtime type → A100 GPU**, reconnect, and run the single code cell. When Colab asks, authorize the Drive account that contains `MyDrive/Quantonium/PhiState-L4`.

In prior CPU reference checks, a dense model improved its next-token predictions on natural code and text. A separate curriculum emphasizing the answer-changing token learned a narrow synthetic task. Those results are included in the notebook. The journal now measures ordinary next-token training on the A100. You can run it directly on your existing Drive project. The cell runs a CUDA operation on the attached A100 before mounting Drive. It launches PyTorch training with matched next-token prediction windows from your repaired **code and general-text** corpus, compares PhiState and a dense attention reference, and uses the measured results to decide whether the archived PhiState base checkpoint should receive a bounded continuation. It verifies the source weights and corpus in the worker. The source checkpoint and the completed chat experiment remain in their existing folders.

The worker verifies the archived 65K and 66K base checkpoints. It runs two independent seeds. For each seed the new PhiState and dense reference models receive the same training windows, 512 updates and 524,288 target tokens per model. A 512-update continuation from the archived 66K base is conditional on both architectures meeting the validation criteria on both seeds. The final report records the decision. [Pythia, matched data order](https://arxiv.org/abs/2304.01373) · [Chinchilla, compute and training-token budgets](https://arxiv.org/abs/2203.15556) · [Griffin, recurrence and local attention](https://arxiv.org/abs/2402.19427).

The worker streams progress and saves its checkpoints and `report.json` under `whitespace_repair_v1/phistate_a100_base_v1/`. To resume after a disconnect, reconnect to an A100 and run this cell again. Only enable `RECOVER_DISCONNECTED` after the previous runtime has ended and the worker reports a stale lock. Leave `RELEASE_GPU_AFTER_FINISH` off if Wan or another job shares this runtime. Enabling it releases the entire Colab runtime only after a complete report has been saved.

In [ ]:
#@title Train and compare base language models on A100
RELEASE_GPU_AFTER_FINISH = False #@param {type:"boolean"}
RECOVER_DISCONNECTED = False #@param {type:"boolean"}

import base64
import hashlib
import importlib.util
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import tempfile
import zlib
from pathlib import Path

PIN = "73edfa1a704e8d198269351ebe597e5ff8fed8ed"
CORPUS_FINGERPRINT = "5a45ae09dfc42c6021202838439e77bd5cb05ff38d673d23e3219a1ecbc6c1d3"
REPO = "https://github.com/Anharmoniclabs/quantoniumoslinux.git"
LAB = Path("runtime/experiments/phi_runtime_lab")
MOUNT = Path("/content/drive")
ROOT = MOUNT / "MyDrive/Quantonium/PhiState-L4/whitespace_repair_v1"
ENTRY = "phistate_a100_base_journal.py"
REPORT = ROOT / "phistate_a100_base_v1/report.json"

def require_a100():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU is attached. Choose A100 GPU under Runtime → Change runtime type, reconnect, and run this cell.")
    gpu = torch.cuda.get_device_properties(0)
    if "A100" not in gpu.name.upper():
        raise RuntimeError(f"Attached GPU is {gpu.name}. Select A100, reconnect, and rerun the cell.")
    with torch.inference_mode():
        value = torch.tensor([2., 3.], device="cuda:0").square().sum()
        torch.cuda.synchronize(0)
        if value.device.type != "cuda" or value.item() != 13.:
            raise RuntimeError("CUDA arithmetic check failed on the selected A100.")
    print(f"{gpu.name} | {gpu.total_memory / (1024**3):.1f} GiB | PyTorch {torch.__version__} | CUDA {torch.version.cuda} | CUDA operation passed", flush=True)

def verify_corpus():
    directory = ROOT / "corpus_repaired_v1"
    manifest = directory / "manifest.json"
    if not manifest.is_file():
        raise FileNotFoundError(f"Repaired base corpus manifest missing from Drive: {manifest}")
    record = json.loads(manifest.read_text(encoding="utf-8"))
    if record.get("format") != "PhiMoECorpus-v1" or record.get("data_fingerprint") != CORPUS_FINGERPRINT:
        raise RuntimeError(f"Unexpected repaired corpus identity at {manifest}")
    print("Found repaired base corpus |", CORPUS_FINGERPRINT, flush=True)

def checkout(stage):
    def git(*args):
        return subprocess.check_output(["git", "-C", str(stage), *args], text=True).strip()
    git("init", "-q")
    git("remote", "add", "origin", REPO)
    git("config", "remote.origin.promisor", "true")
    git("config", "remote.origin.partialclonefilter", "blob:none")
    git("sparse-checkout", "set", "--no-cone", "/runtime/experiments/phi_runtime_lab/", "/runtime/src/rftmw_native/")
    git("fetch", "--quiet", "--depth=1", "--filter=blob:none", "origin", PIN)
    git("checkout", "--quiet", "--detach", "FETCH_HEAD")
    lab = stage / LAB
    if git("rev-parse", "HEAD") != PIN or not (lab / "moe_train.py").is_file():
        raise RuntimeError("Public PhiState checkout could not be verified.")
    return lab

def read_package():
    packed = zlib.decompress(base64.b64decode(SOURCES, validate=True))
    if hashlib.sha256(packed).hexdigest() != PACKAGE_SHA:
        raise RuntimeError("Embedded experiment package did not pass its integrity check.")
    modules = json.loads(packed)
    expected = hashlib.sha256(modules["phistate_base_prerequisite.py"].encode()).hexdigest()
    if expected != PREREQUISITE_SOURCE_HASH:
        raise RuntimeError("Embedded prerequisite decision differs from the reference receipt.")
    if ENTRY not in modules:
        raise RuntimeError("Embedded experiment entry point missing.")
    return modules

def unpack_scripts(lab, modules):
    for name, source in modules.items():
        if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.py", name):
            raise RuntimeError("Invalid embedded module filename.")
        compile(source, name, "exec")
        path = lab / name
        if name == "device_preflight.py":
            pinned_hash = "8ecd69302002cabc318f56cedb1c2b82647ee1343b58f69faaca3554ef24450d"
            if hashlib.sha256(path.read_bytes()).hexdigest() != pinned_hash:
                raise RuntimeError("Pinned base preflight dependency has changed.")
            mode = "wb"
        else:
            mode = "xb"
        with path.open(mode) as stream:
            stream.write(source.encode("utf-8"))

def run_worker(lab):
    env = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONDONTWRITEBYTECODE="1",
               TOKENIZERS_PARALLELISM="false", PHISTATE_JOURNAL_COMMIT=PIN,
               PYTHONPATH=os.pathsep.join([str(lab), *filter(None, [os.environ.get("PYTHONPATH")])]))
    args = [sys.executable, "-u", str(lab / ENTRY), "--root", str(ROOT)]
    if RECOVER_DISCONNECTED:
        args.append("--recover-disconnected")
    interrupted = False
    with subprocess.Popen(args, cwd=lab, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1, start_new_session=True) as child:
        with child.stdout:
            while True:
                try:
                    for line in child.stdout:
                        print(line, end="", flush=True)
                    code = child.wait()
                    break
                except KeyboardInterrupt:
                    if not interrupted:
                        interrupted = True
                        print("Stop requested. Waiting for the worker to exit.", flush=True)
                        try:
                            child.send_signal(signal.SIGINT)
                        except ProcessLookupError:
                            pass
    if code and not (interrupted and code in (-signal.SIGINT, 130)):
        raise RuntimeError(f"Base training worker exited with status {code}. See the last worker message above.")
    return interrupted

def launch():
    modules = read_package()
    if not PREREQUISITE_QUALIFIED:
        raise RuntimeError("Reference qualification is incomplete for this snapshot. Leave the A100 disconnected; no Drive mount, CUDA probe, or training has been requested.")
    require_a100()
    if not (MOUNT / "MyDrive").is_dir():
        from google.colab import drive
        drive.mount(str(MOUNT))
    verify_corpus()
    missing = [m for m in ("numpy", "sentencepiece") if importlib.util.find_spec(m) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
    with tempfile.TemporaryDirectory(prefix="phistate_a100_base_", dir="/content") as tmp:
        lab = checkout(Path(tmp))
        unpack_scripts(lab, modules)
        print("Starting | base", PIN, "| experiment", PACKAGE_SHA[:16], "| report", REPORT, flush=True)
        if run_worker(lab):
            print("Worker stopped. Reconnect to A100 and rerun to resume.", flush=True)
            return
    if not REPORT.is_file():
        raise RuntimeError(f"Worker ended without its expected Drive report: {REPORT}")
    status = json.loads(REPORT.read_text(encoding="utf-8"))
    print(f"Drive report: {REPORT}", flush=True)
    for key in ("status", "outcome", "conclusion", "next_action"):
        if key in status:
            print(f"{key}: {status[key]}", flush=True)

    if RELEASE_GPU_AFTER_FINISH and status.get("status") == "complete":
        print("Verified completed Drive report. Releasing this Colab runtime.", flush=True)
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception as exc:
            print(f"Use Runtime → Disconnect and delete runtime to release the GPU ({exc}).", flush=True)

# Compressed UTF-8 experiment sources; verified before running.
PACKAGE_SHA = "7d484ca6dcca1296cd62b44629c1ca5253cc5e0c2373e3bd9ef4017035d04d5b"
PREREQUISITE_SOURCE_HASH = "cdba04e0c1e752dd60d7ebf90295e73dd598ebc3b63da685cd97a4fd213cc2e9"
PREREQUISITE_QUALIFIED = True
SOURCES = (
    '...'
)
launch()
